***I will train 2 Logistic Regression models where Model 2 accounts for the moment in time that the pass is made during the match, and Model 1 doesn't***

In [1]:
import pandas as pd

df = pd.read_csv('../data/processed/final_training_corpus.csv')

df.columns

Index(['match_id', 'period', 'half_percentage', 'net_score', 'under_pressure',
       'start_x', 'start_y', 'end_x', 'end_y', 'pass_angle', 'pass_length',
       'dist_to_goal', 'dist_receiv_middle', 'pass_outcome'],
      dtype='str')

In [2]:
features1 = [ 
    'under_pressure',
    'start_x', 
    'start_y', 
    'end_x', 
    'end_y', 
    'pass_angle', 
    'pass_length',
    'dist_to_goal', 
    'dist_receiv_middle'
]

In [3]:
features2 = [
    'period', 
    'half_percentage', 
    'net_score', 
    'under_pressure',
    'start_x', 
    'start_y', 
    'end_x', 
    'end_y', 
    'pass_angle', 
    'pass_length',
    'dist_to_goal', 
    'dist_receiv_middle'
]

In [4]:
X1 = df[features1]
y1 = df['pass_outcome']

In [5]:
X2 = df[features2]
y2 = df['pass_outcome']

In [6]:
# Divide my data to training and test with an 80% to 20% split respectively
from sklearn.model_selection import train_test_split
X_train1, X_test1, y_train1, y_test1 = train_test_split(X1, y1, test_size = 0.2, random_state = 42)

In [7]:
from sklearn.model_selection import train_test_split
X_train2, X_test2, y_train2, y_test2 = train_test_split(X2, y2, test_size = 0.2, random_state = 42)

In [8]:
print(f"Training Passes: {len(X_train1)} passes")
print(f"Testing Passes: {len(X_test1)} passes")

Training Passes: 32269 passes
Testing Passes: 8068 passes


In [9]:
# Prevent jupyter notebook html display errors
import sklearn
sklearn.set_config(display="text")

In [10]:
# Firstly I will train a Logistic Regression model

from sklearn.linear_model import LogisticRegression

log_reg_model1 = LogisticRegression(max_iter=1000)
log_reg_model1.fit(X_train1, y_train1)

LogisticRegression(max_iter=1000)

In [11]:
from sklearn.linear_model import LogisticRegression

log_reg_model2 = LogisticRegression(max_iter=1000)
log_reg_model2.fit(X_train2, y_train2)

LogisticRegression(max_iter=1000)

In [12]:
from sklearn.metrics import accuracy_score

log_reg_train_predictions1 = log_reg_model1.predict(X_train1)
log_reg_test_predictions1 = log_reg_model1.predict(X_test1)

print(f"Training Accuracy for Model 1: {accuracy_score(y_train1, log_reg_train_predictions1)*100:.4f} %")
print(f"Testing Accuracy for Model 1: {accuracy_score(y_test1, log_reg_test_predictions1)*100:.4f} %")

Training Accuracy for Model 1: 86.8233 %
Testing Accuracy for Model 1: 86.5394 %


In [13]:
from sklearn.metrics import accuracy_score

log_reg_train_predictions2 = log_reg_model2.predict(X_train2)
log_reg_test_predictions2 = log_reg_model2.predict(X_test2)

print(f"Training Accuracy for Model 2: {accuracy_score(y_train2, log_reg_train_predictions2)*100:.4f} %")
print(f"Testing Accuracy for Model 2: {accuracy_score(y_test2, log_reg_test_predictions2)*100:.4f} %")

Training Accuracy for Model 2: 86.8388 %
Testing Accuracy for Model 2: 86.5518 %


In [14]:
#Calculate Probabilistic Metrics for Model 1

from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score

log_reg_y_prob1 = log_reg_model1.predict_proba(X_test1)[:, 1]

log_reg_baseline_log_loss1 = log_loss(y_test1, log_reg_y_prob1)
log_reg_baseline_brier1 = brier_score_loss(y_test1, log_reg_y_prob1)
log_reg_baseline_auc1 = roc_auc_score(y_test1, log_reg_y_prob1)

print(f"Baseline Log-Loss for Model 1: {log_reg_baseline_log_loss1:.5f}")
print(f"Baseline Brier Score for Model 1: {log_reg_baseline_brier1:.5f}")
print(f"Baseline ROC-AUC for Model 1: {log_reg_baseline_auc1:.5f}")

Baseline Log-Loss for Model 1: 0.33189
Baseline Brier Score for Model 1: 0.10045
Baseline ROC-AUC for Model 1: 0.79985


In [15]:
#Calculate Probabilistic Metrics for Model 2

from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score

log_reg_y_prob2 = log_reg_model2.predict_proba(X_test2)[:, 1]

log_reg_baseline_log_loss2 = log_loss(y_test2, log_reg_y_prob2)
log_reg_baseline_brier2 = brier_score_loss(y_test2, log_reg_y_prob2)
log_reg_baseline_auc2 = roc_auc_score(y_test2, log_reg_y_prob2)

print(f"Baseline Log-Loss for Model 2: {log_reg_baseline_log_loss2:.5f}")
print(f"Baseline Brier Score for Model 2: {log_reg_baseline_brier2:.5f}")
print(f"Baseline ROC-AUC for Model 2: {log_reg_baseline_auc2:.5f}")

Baseline Log-Loss for Model 2: 0.33179
Baseline Brier Score for Model 2: 0.10049
Baseline ROC-AUC for Model 2: 0.80020


***I will now train and compare in a similar way 2 Models using XGBOOST***

In [16]:
import xgboost as xgb

# Specify the parameters of my models
xgb_config = {
    'n_estimators': 100,
    'max_depth': 4,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'eval_metric': 'logloss'
}

xgb1 = xgb.XGBClassifier(**xgb_config)
xgb2 = xgb.XGBClassifier(**xgb_config)

In [17]:
# Train the two models
print('Training XGBOOST Model 1...')
xgb1.fit(X_train1, y_train1)
print('XGBOOST Model 1 training Done!')

print('Training XGBOOST Model 2...')
xgb2.fit(X_train2, y_train2)
print('XGBOOST Model 2 training Done!')

Training XGBOOST Model 1...
XGBOOST Model 1 training Done!
Training XGBOOST Model 2...
XGBOOST Model 2 training Done!


In [18]:
xgb_train_predictions1 = xgb1.predict(X_train1)
xgb_test_predictions1 = xgb1.predict(X_test1)

print(f"Training Accuracy for XGBOOST Model 1: {accuracy_score(y_train2, xgb_train_predictions1)*100:.4f} %")
print(f"Testing Accuracy for XGBOOST Model 1: {accuracy_score(y_test2, xgb_test_predictions1)*100:.4f} %")

xgb_train_predictions2 = xgb2.predict(X_train2)
xgb_test_predictions2 = xgb2.predict(X_test2)

print(f"Training Accuracy for XGBOOST Model 2: {accuracy_score(y_train2, xgb_train_predictions2)*100:.4f} %")
print(f"Testing Accuracy for XGBOOST Model 2: {accuracy_score(y_test2, xgb_test_predictions2)*100:.4f} %")

Training Accuracy for XGBOOST Model 1: 90.4738 %
Testing Accuracy for XGBOOST Model 1: 89.8240 %
Training Accuracy for XGBOOST Model 2: 90.4397 %
Testing Accuracy for XGBOOST Model 2: 89.7992 %


In [19]:
#Calculate Probabilistic Metrics for XGBOOST Model 1

xgb_y_prob1 = xgb1.predict_proba(X_test1)[:, 1]

xgb_baseline_log_loss1 = log_loss(y_test1, xgb_y_prob1)
xgb_baseline_brier1 = brier_score_loss(y_test1, xgb_y_prob1)
xgb_baseline_auc1 = roc_auc_score(y_test1, xgb_y_prob1)

print(f"Baseline Log-Loss for XGBOOST Model 1: {xgb_baseline_log_loss1:.5f}")
print(f"Baseline Brier Score for XGBOOST Model 1: {xgb_baseline_brier1:.5f}")
print(f"Baseline ROC-AUC for XGBOOST Model 1: {xgb_baseline_auc1:.5f}")

Baseline Log-Loss for XGBOOST Model 1: 0.26588
Baseline Brier Score for XGBOOST Model 1: 0.07819
Baseline ROC-AUC for XGBOOST Model 1: 0.88440


In [20]:
#Calculate Probabilistic Metrics for XGBOOST Model 1

xgb_y_prob2 = xgb2.predict_proba(X_test2)[:, 1]

xgb_baseline_log_loss2 = log_loss(y_test2, xgb_y_prob2)
xgb_baseline_brier2 = brier_score_loss(y_test2, xgb_y_prob2)
xgb_baseline_auc2 = roc_auc_score(y_test2, xgb_y_prob2)

print(f"Baseline Log-Loss for XGBOOST Model 2: {xgb_baseline_log_loss2:.5f}")
print(f"Baseline Brier Score for XGBOOST Model 2: {xgb_baseline_brier2:.5f}")
print(f"Baseline ROC-AUC for XGBOOST Model 2: {xgb_baseline_auc2:.5f}")

Baseline Log-Loss for XGBOOST Model 2: 0.26720
Baseline Brier Score for XGBOOST Model 2: 0.07855
Baseline ROC-AUC for XGBOOST Model 2: 0.88277
